# Pipeline RAG Ngân hàng (Dual-Indexing)
Pipeline gồm 5 bước: Query Parser -> Hybrid Retrieval (RRF) -> Reranking -> Context Synthesizer -> LLM Serving.

In [1]:
import os
import sys
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

load_dotenv(ROOT_DIR / ".env")

from rag_banking.config import Config
from rag_banking.pipeline import BankingRAGPipeline

print("Đã tải thư viện thành công.")


Đã tải thư viện thành công.


## Khởi tạo Pipeline & Lập chỉ mục kép (Dual-Indexing)
Nạp tài liệu quy chế, phân đoạn breadcrumbs, tạo vector Dense (1024 chiều) và chỉ mục từ khóa Okapi BM25.

In [2]:
# Khởi tạo pipeline
pipeline = BankingRAGPipeline()

# Danh sách tài liệu ngân hàng mẫu
doc_files = [
    "phap_ly__quy_che_ekyc_vietcombank_han_muc.md",
    "phap_ly__dieu_kien_dieu_khoan_mo_tai_khoan_truc_tuyen_ekyc.md",
    "phap_ly__dieu_kien_dieu_khoan_mo_va_su_dung_tai_khoan_khcn.md",
    "quy_trinh__mo_tai_khoan_thanh_toan_ekyc_va_xac_thuc_nfc.md",
    "faq__ngan_hang_so_vcb_digibank_va_smart_otp.md",
    "ngan_hang_so__quy_chuan_dich_vu_digibank_va_han_muc.md",
    "tai_khoan__bieu_phi_dich_vu_tai_khoan_thanh_toan.md",
    "cho_vay__vay_tieu_dung_co_tai_san_bao_dam.md"
]

all_chunks = []
for fname in doc_files:
    fpath = Config.RETRIEVAL_DATASET_DIR / fname
    if fpath.exists():
        text = fpath.read_text(encoding="utf-8", errors="ignore")
        chunks = pipeline.chunker.chunk_text(text, source_name=fname)
        all_chunks.extend(chunks)

# Lập chỉ mục: Dense Vector (1024-d) + Sparse BM25
texts = [c["content"] for c in all_chunks]
vectors = pipeline.embedding_model.embed_documents(texts, batch_size=32)
pipeline.vector_store.add_chunks(all_chunks, vectors)
pipeline.bm25_search.index(all_chunks)
pipeline.indexed_chunks = all_chunks

print(f"Đã lập chỉ mục xong {len(all_chunks)} chunks.")


Đã lập chỉ mục xong 151 chunks.


## Bước 1: Query Parser
Chuẩn hóa từ viết tắt (`eKYC`, `TKTT`, `Smart OTP`, `VCB Digibank`) và trích xuất từ khóa.

In [3]:
query = "Hạn mức giao dịch và điều kiện mở TKTT trực tuyến qua eKYC trên app VCB Digibank xác thực Smart OTP là gì?"
parsed = pipeline.step1_query_parser(query)

print("Câu gốc:", parsed.raw_query)
print("Chuẩn hóa:", parsed.normalized_query)
print("Từ viết tắt:", parsed.detected_acronyms)
print("Từ khóa BM25:", parsed.keywords[:6])


Câu gốc: Hạn mức giao dịch và điều kiện mở TKTT trực tuyến qua eKYC trên app VCB Digibank xác thực Smart OTP là gì?
Chuẩn hóa: Hạn mức giao dịch và điều kiện mở tài khoản thanh toán (TKTT) trực tuyến qua định danh điện tử (eKYC) trên app ngân hàng số VCB Digibank (Vietcombank Digibank) xác thực phương thức xác thực Smart OTP là gì?
Từ viết tắt: [('VCB Digibank', 'ngân hàng số VCB Digibank (Vietcombank Digibank)'), ('Smart OTP', 'phương thức xác thực Smart OTP'), ('eKYC', 'định danh điện tử (eKYC)'), ('TKTT', 'tài khoản thanh toán (TKTT)')]
Từ khóa BM25: ['ekyc', 'tktt', 'smart otp', 'vcb digibank', 'hạn', 'mức']


## Bước 2: Hybrid Retrieval
Tìm kiếm đồng thời trên Dense Vector và BM25, hợp nhất bằng RRF (k=60) lấy Top-5.

In [4]:
top5 = pipeline.step2_hybrid_retrieval(parsed, top_k=5)

for c in top5:
    print(f"Top {c['candidate_rank']} | RRF: {c['rrf_score']:.4f} | Nguồn: {c['chunk'].get('source')} | Thẻ: {c['chunk'].get('legal_ref')}")


Top 1 | RRF: 0.0160 | Nguồn: phap_ly__quy_che_ekyc_vietcombank_han_muc.md | Thẻ: [Điều 12, Khoản 1]
Top 2 | RRF: 0.0160 | Nguồn: quy_trinh__mo_tai_khoan_thanh_toan_ekyc_va_xac_thuc_nfc.md | Thẻ: 
Top 3 | RRF: 0.0154 | Nguồn: faq__ngan_hang_so_vcb_digibank_va_smart_otp.md | Thẻ: 
Top 4 | RRF: 0.0150 | Nguồn: faq__ngan_hang_so_vcb_digibank_va_smart_otp.md | Thẻ: 
Top 5 | RRF: 0.0149 | Nguồn: phap_ly__quy_che_ekyc_vietcombank_han_muc.md | Thẻ: [Điều 12, Khoản 2; Quyết định 2345/QĐ-NHNN]


## Bước 3: Reranking
So khớp câu hỏi với câu hỏi chuẩn và độ trùng lặp từ vựng để chọn Top-3 chunks.

In [5]:
top3 = pipeline.step3_reranking(parsed, top5, top_k=3)

for c in top3:
    print(f"Top {c['rerank_rank']} | Score: {c['final_rerank_score']:.4f} | Nguồn: {c['chunk'].get('source')}")


Top 1 | Score: 0.6449 | Nguồn: phap_ly__quy_che_ekyc_vietcombank_han_muc.md
Top 2 | Score: 0.6247 | Nguồn: quy_trinh__mo_tai_khoan_thanh_toan_ekyc_va_xac_thuc_nfc.md
Top 3 | Score: 0.6190 | Nguồn: faq__ngan_hang_so_vcb_digibank_va_smart_otp.md


## Bước 4: Context Synthesizer
Đóng gói ngữ cảnh Markdown, gắn thẻ pháp lý [Điều X, Khoản Y] và kiểm soát giới hạn token.

In [6]:
context = pipeline.step4_context_synthesizer(top3, max_tokens=2000)

print("Số token:", context.total_tokens)
print("Thẻ pháp lý:", context.legal_citations)
print("Nguồn:", context.sources_used)
print("\nNgữ cảnh gửi LLM (trích đoạn):\n", context.markdown_context[:400], "...")


Số token: 725
Thẻ pháp lý: ['[Điều 12, Khoản 1]', '[Điều 12]']
Nguồn: ['faq__ngan_hang_so_vcb_digibank_va_smart_otp.md', 'phap_ly__quy_che_ekyc_vietcombank_han_muc.md', 'quy_trinh__mo_tai_khoan_thanh_toan_ekyc_va_xac_thuc_nfc.md']

Ngữ cảnh gửi LLM (trích đoạn):
### [TRÍCH DẪN 1] Nguồn: phap_ly__quy_che_ekyc_vietcombank_han_muc.md
- **Tham chiếu pháp lý:** `[Điều 12, Khoản 1], [Điều 12]`
- **Đường dẫn phân cấp (Breadcrumbs):** `QUY CHẾ MỞ VÀ SỬ DỤNG TÀI KHOẢN THANH TOÁN TRỰC TUYẾN EKYC VIETCOMBANK > CHƯƠNG III: QUY ĐỊNH HẠN MỨC GIAO DỊCH VÀ XÁC THỰC SINH TRẮC HỌC > Điều 12. Hạn mức giao dịch tài khoản thanh toán mở qua eKYC`
- **Nội dung xác thực:**
1. Kh ...


## Bước 5: LLM Serving
Truyền ngữ cảnh và Banking Guardrails vào LLM (DeepSeek-V4-Flash) để sinh câu trả lời.

In [7]:
serving = pipeline.step5_llm_serving(parsed.raw_query, context)

print("Mô hình:", serving["model_served"])
print("\nCâu trả lời của bot:\n", serving["answer"])


Mô hình: DeepSeek-V4-Flash

Câu trả lời của bot:
Kính thưa Quý khách,

Vietcombank xin trả lời Quý khách về hạn mức giao dịch và điều kiện mở tài khoản thanh toán trực tuyến qua eKYC trên ứng dụng VCB Digibank như sau:

**1. Điều kiện mở tài khoản thanh toán trực tuyến qua eKYC:**
- Khách hàng cá nhân mở tài khoản thanh toán trực tuyến qua phương thức điện tử (eKYC) trên ứng dụng VCB Digibank sẽ được áp dụng hạn mức giao dịch theo từng cấp độ xác thực nhằm bảo đảm an toàn tài chính `[Điều 12, Khoản 1]`.

**2. Hạn mức giao dịch khi mở eKYC trực tuyến (chưa xác thực tại quầy):**
- **Hạn mức chuyển tiền tối đa:** 100.000.000 VNĐ/tháng (theo Thông tư 16/2020/TT-NHNN) `[Quy chuẩn nghiệp vụ]`.
- Phương thức xác thực Smart OTP được tích hợp trong ứng dụng VCB Digibank, tuy nhiên hạn mức này vẫn áp dụng theo cấp độ xác thực của tài khoản eKYC chưa nâng hạn mức.

**3. Cách nâng hạn mức lên đến 03 tỷ VNĐ/ngày:**
- Quý khách mang **CCCD gắn chip** đến quầy giao dịch Vietcombank để xác thực chữ ký

## 1.4. Kiểm thử End-to-End: Hạn mức mở tài khoản eKYC
Câu hỏi nghiệp vụ: **"Hạn mức mở tài khoản eKYC tối đa bao nhiêu 1 ngày?"**

In [8]:
query_14 = "Hạn mức mở tài khoản eKYC tối đa bao nhiêu 1 ngày?"
res_14 = pipeline.run_pipeline(query_14)

print("Câu hỏi:", query_14)
print("\nCâu trả lời của bot:\n", res_14["answer"])
print("\nDẫn chứng nguồn (Citations):", res_14["legal_citations"])

# Bảng độ trễ
lat = res_14["latency"]
df_lat = pd.DataFrame([
    {"Bước": "1. Query Parser", "Độ trễ (ms)": f"{lat['step1_query_parser_ms']:.1f}"},
    {"Bước": "2. Hybrid Retrieval (RRF)", "Độ trễ (ms)": f"{lat['step2_hybrid_retrieval_ms']:.1f}"},
    {"Bước": "3. Reranking", "Độ trễ (ms)": f"{lat['step3_reranking_ms']:.1f}"},
    {"Bước": "4. Context Synthesizer", "Độ trễ (ms)": f"{lat['step4_context_synthesizer_ms']:.1f}"},
    {"Bước": "5. LLM Serving", "Độ trễ (ms)": f"{lat['step5_llm_serving_ms']:.1f}"},
    {"Bước": "Tổng thời gian", "Độ trễ (ms)": f"{lat['total_pipeline_ms']:.1f} ({lat['total_pipeline_ms']/1000:.2f}s)"}
])
display(df_lat)


Câu hỏi: Hạn mức mở tài khoản eKYC tối đa bao nhiêu 1 ngày?

Câu trả lời của bot:
Kính thưa Quý khách,

Theo quy định hiện hành của Vietcombank, hạn mức giao dịch tối đa trong ngày đối với tài khoản thanh toán mở qua eKYC được phân định như sau:

| Loại hạn mức | Hạn mức tối đa/ngày | Điều kiện |
|---|---|---|
| **Hạn mức thông thường** | **100.000.000 VNĐ/ngày** | Hoàn tất eKYC cơ bản |
| **Hạn mức nâng cao** | **400.000.000 VNĐ/ngày** | Hoàn tất xác thực sinh trắc học CCCD gắn chip qua NFC theo Quyết định 2345/QĐ-NHNN |

Căn cứ pháp lý: **[Điều 12, Khoản 2, Quyết định 2345/QĐ-NHNN]**.

Ngoài ra, nếu Quý khách có nhu cầu giao dịch vượt quá **400.000.000 VNĐ/ngày**, Quý khách vui lòng mang theo bản chính CCCD gắn chip đến điểm giao dịch Vietcombank gần nhất để đăng ký chữ ký mẫu và xác thực trực tiếp tại quầy, theo **[Điều 12, Khoản 3]**.

Lưu ý thêm: Mọi giao dịch chuyển tiền từ **10.000.000 VNĐ/lần** trở lên hoặc tổng giá trị giao dịch trong ngày vượt quá **20.000.000 VNĐ/ngày** bắt 

Bước,Độ trễ (ms)
1. Query Parser,0.2
2. Hybrid Retrieval (RRF),149.0
3. Reranking,235.5
4. Context Synthesizer,0.2
5. LLM Serving,11109.2
Tổng thời gian,11494.1 (11.49s)


## Thử nghiệm thêm các câu hỏi khác


In [9]:
queries = [
    "Smart OTP là gì và điều kiện để kích hoạt phương thức xác thực Smart OTP trên app VCB Digibank?",
    "Phí mở tài khoản thanh toán và số dư tối thiểu yêu cầu tại Vietcombank là bao nhiêu?"
]

for q in queries:
    print("=" * 60)
    print("Câu hỏi:", q)
    res = pipeline.run_pipeline(q)
    print("\nCâu trả lời của bot:\n", res["answer"])
    print(f"\nThời gian phản hồi: {res['latency']['total_pipeline_ms']:.1f} ms\n")


Câu hỏi: Smart OTP là gì và điều kiện để kích hoạt phương thức xác thực Smart OTP trên app VCB Digibank?

Câu trả lời của bot:
Kính thưa Quý khách,

**VCB Smart OTP** là phương thức xác thực giao dịch trong đó mã OTP được sinh ra bởi ứng dụng được cài đặt trên thiết bị di động của Quý khách. Đối với dịch vụ VCB Digibank trên thiết bị di động, Smart OTP được tích hợp ngay trong ứng dụng.

Để kích hoạt VCB Smart OTP, Quý khách vui lòng thực hiện các bước sau:

- Tải ứng dụng **VCB Smart OTP** và nhập **mã kích hoạt ứng dụng** do Vietcombank gửi đến số điện thoại đăng ký VCB Smart OTP.
- Thực hiện **02 giao dịch tài chính** (chuyển tiền, thanh toán, nạp tiền, tiết kiệm…) có xác thực bằng **SMS OTP** trên VCB Digibank.
- Sau khi hoàn tất 02 giao dịch này, Quý khách có thể bắt đầu sử dụng VCB Smart OTP để xác thực giao dịch.

Vietcombank khuyến cáo Quý khách **không chia sẻ mã OTP, mật khẩu hoặc thông tin bảo mật** cho bất kỳ ai. Nếu cần hỗ trợ thêm, Quý khách vui lòng liên hệ Tổng đài **19